# FloodNet U-Net Training — Road Waterlogging Segmentation

**Severity Methodology:**  
All severity tiers are *Project-defined rule-based severity tiers derived from the predicted flooded-road area percentage.*  
FloodNet does NOT provide ground-truth severity labels.

**Classes of interest:**
- Class 3 = Road-flooded
- Class 4 = Road-non-flooded  
- Class 5 = Water

**Severity thresholds:**
- ≤ 0.5% → NONE | 0.5–15% → LOW | 15–40% → MODERATE | >40% → SEVERE

In [ ]:
# ── Step 1: Check GPU ──────────────────────────────────────────────────────
import subprocess
result = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
print(result.stdout if result.returncode == 0 else 'No GPU detected')

import torch
print(f'PyTorch: {torch.__version__}')
print(f'CUDA available: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')

In [ ]:
# ── Step 2: Upload FloodNet dataset ───────────────────────────────────────
# OPTION A — Google Drive (recommended for Colab)
from google.colab import drive
drive.mount('/content/drive')

# OPTION B — Direct upload (for Kaggle or if not using Drive)
# from google.colab import files
# uploaded = files.upload()  # upload FloodNet.zip here

# After mounting Drive, set the path to your FloodNet zip:
FLOODNET_ZIP = '/content/drive/MyDrive/FloodNet.zip'  # <-- CHANGE THIS PATH
print(f'FloodNet zip path set to: {FLOODNET_ZIP}')

In [ ]:
# ── Step 3: Extract dataset ────────────────────────────────────────────────
import zipfile, os
from pathlib import Path

EXTRACT_DIR = Path('/content/floodnet_dataset')
EXTRACT_DIR.mkdir(exist_ok=True)

print(f'Extracting {FLOODNET_ZIP} ...')
with zipfile.ZipFile(FLOODNET_ZIP, 'r') as z:
    z.extractall(EXTRACT_DIR)

# Verify structure
for split in ['train', 'val', 'test']:
    for sub in [f'{split}-org-img', f'{split}-label-img']:
        p = EXTRACT_DIR / 'FloodNet-Supervised_v1.0' / split / sub
        count = len(list(p.glob('*'))) if p.exists() else 0
        print(f'  {p.relative_to(EXTRACT_DIR)}  → {count} files')

FLOODNET_DIR = EXTRACT_DIR / 'FloodNet-Supervised_v1.0'
print(f'\nFloodNet root: {FLOODNET_DIR}')

In [ ]:
# ── Step 4: Upload project source code ────────────────────────────────────
# Upload Road_Waterlogging_Project.zip (zip your src/ folder)
# OR clone from GitHub if you have it there

# Option A — Upload zip:
# from google.colab import files
# files.upload()  # upload Road_Waterlogging_Project.zip

# Option B — Copy from Drive:
PROJECT_ZIP = '/content/drive/MyDrive/Road_Waterlogging_Project.zip'  # <-- CHANGE
!unzip -q "{PROJECT_ZIP}" -d /content/

import sys
sys.path.insert(0, '/content/Road_Waterlogging_Project')

# Verify imports
from src.config import FLOODNET_CLASSES, NUM_CLASSES
from src.dataset_loader import build_floodnet_dataloaders
from src.segmentation import UNet, CombinedSegmentationLoss, compute_floodnet_multiclass_metrics
print('All imports OK')
print(f'Classes: {NUM_CLASSES} | {list(FLOODNET_CLASSES.values())}')

In [ ]:
# ── Step 5: Install dependencies ──────────────────────────────────────────
!pip install -q scikit-learn tqdm
print('Dependencies ready')

In [ ]:
# ── Step 6: Training configuration ────────────────────────────────────────
import torch

CONFIG = {
    'floodnet_dir'    : FLOODNET_DIR,
    'epochs'          : 25,            # More epochs on GPU
    'batch_size'      : 8,             # Increase to 16 if VRAM > 12 GB
    'lr'              : 1e-3,
    'weight_decay'    : 1e-4,
    'patience'        : 7,
    'img_size'        : (256, 256),
    'base_features'   : 64,            # Full 31M-param U-Net on GPU
    'num_classes'     : 10,
    'device'          : torch.device('cuda' if torch.cuda.is_available() else 'cpu'),
    'save_dir'        : Path('/content/models'),
    'reports_dir'     : Path('/content/reports'),
    'severity_method' : 'Project-defined rule-based severity tiers derived from the predicted flooded-road area percentage',
}
CONFIG['save_dir'].mkdir(exist_ok=True)
CONFIG['reports_dir'].mkdir(exist_ok=True)

print('Training Configuration:')
for k, v in CONFIG.items():
    print(f'  {k:<20s}: {v}')

In [ ]:
# ── Step 7: Build dataloaders ──────────────────────────────────────────────
loaders = build_floodnet_dataloaders(
    floodnet_dir=CONFIG['floodnet_dir'],
    batch_size=CONFIG['batch_size'],
    img_size=CONFIG['img_size'],
    num_workers=2,  # GPU environment can use workers
    include_test=False,  # Test split is optional / kept locally
)
counts = loaders['split_counts']
print(f"Train: {counts['train']} | Val: {counts['val']} | Test: {counts['test']} (optional)")


In [ ]:
# ── Step 8: Class-weight estimation ───────────────────────────────────────
import numpy as np
from src.config import FLOODNET_CLASSES

def compute_class_weights(train_loader, num_classes=10, max_batches=100, device='cpu'):
    print(f'Estimating class weights from up to {max_batches} batches ...')
    counts = torch.zeros(num_classes, dtype=torch.float64)
    for i, batch in enumerate(train_loader):
        if i >= max_batches:
            break
        _, masks, *_ = batch
        for c in range(num_classes):
            counts[c] += (masks == c).sum().item()
    total = counts.sum()
    weights = total / (num_classes * counts.clamp(min=1.0))
    weights = weights / weights.sum() * num_classes
    for c, (name, w) in enumerate(zip(FLOODNET_CLASSES.values(), weights.tolist())):
        print(f'  Class {c:2d} {name:<22s}  w={w:.4f}')
    return weights.float().to(device)

class_weights = compute_class_weights(
    loaders['train_loader'], num_classes=10, max_batches=100, device=CONFIG['device']
)

In [ ]:
# ── Step 9: Model, loss, optimiser ────────────────────────────────────────
import torch.optim as optim
from torch.optim.lr_scheduler import ReduceLROnPlateau

model = UNet(in_channels=3, num_classes=CONFIG['num_classes'],
             base_features=CONFIG['base_features']).to(CONFIG['device'])

total_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'U-Net: {total_params:,} trainable parameters')

criterion = CombinedSegmentationLoss(
    class_weights=class_weights, ce_weight=0.5, dice_weight=0.5
)
optimizer = optim.Adam(model.parameters(), lr=CONFIG['lr'], weight_decay=CONFIG['weight_decay'])
scheduler = ReduceLROnPlateau(optimizer, mode='min', factor=0.5, patience=3)

In [ ]:
# ── Step 10: Training loop ─────────────────────────────────────────────────
import json, time
from tqdm.notebook import tqdm

def train_one_epoch(model, loader, criterion, optimizer, device):
    model.train()
    total_loss = 0.0
    pbar = tqdm(loader, desc='Train', leave=False)
    for batch in pbar:
        images, masks, *_ = batch
        images, masks = images.to(device), masks.to(device)
        optimizer.zero_grad()
        loss = criterion(model(images), masks)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        pbar.set_postfix({'loss': f'{loss.item():.4f}'})
    return total_loss / len(loader)

@torch.no_grad()
def validate(model, loader, criterion, device, num_classes=10):
    model.eval()
    total_loss = 0.0
    all_preds, all_targets = [], []
    for batch in tqdm(loader, desc='Val', leave=False):
        images, masks, *_ = batch
        images, masks = images.to(device), masks.to(device)
        logits = model(images)
        total_loss += criterion(logits, masks).item()
        all_preds.append(logits.argmax(dim=1).cpu().numpy())
        all_targets.append(masks.cpu().numpy())
    pred_arr = np.concatenate([p.flatten() for p in all_preds])
    tgt_arr  = np.concatenate([t.flatten() for t in all_targets])
    metrics = compute_floodnet_multiclass_metrics(pred_arr, tgt_arr, num_classes=num_classes)
    metrics['val_loss'] = round(total_loss / len(loader), 6)
    return metrics

# ── Main loop ──────────────────────────────────────────────────────────────
history = []
best_miou = 0.0
patience_counter = 0
EPOCHS = CONFIG['epochs']
PATIENCE = CONFIG['patience']
DEVICE = CONFIG['device']
best_ckpt = CONFIG['save_dir'] / 'floodnet_unet_best.pth'

t_start = time.time()
for epoch in range(1, EPOCHS + 1):
    t0 = time.time()
    train_loss = train_one_epoch(model, loaders['train_loader'], criterion, optimizer, DEVICE)
    val_metrics = validate(model, loaders['val_loader'], criterion, DEVICE)
    val_loss = val_metrics['val_loss']
    miou     = val_metrics['mIoU']
    scheduler.step(val_loss)

    elapsed = time.time() - t0
    print(
        f'Epoch [{epoch:02d}/{EPOCHS}]  '
        f'Train: {train_loss:.4f}  Val: {val_loss:.4f}  '
        f'mIoU: {miou:.4f}  PixAcc: {val_metrics["pixel_accuracy"]:.4f}  '
        f'Road-flooded IoU: {val_metrics["road_flooded_iou"]:.4f}  '
        f'Time: {elapsed:.0f}s'
    )

    history.append({
        'epoch': epoch, 'train_loss': round(train_loss, 6),
        'val_loss': val_loss, 'mIoU': miou,
        'pixel_accuracy': val_metrics['pixel_accuracy'],
        'road_flooded_iou': val_metrics['road_flooded_iou'],
        'road_non_flooded_iou': val_metrics['road_non_flooded_iou'],
        'water_iou': val_metrics['water_iou'],
        'elapsed_s': round(elapsed, 1),
    })

    if miou > best_miou:
        best_miou = miou
        patience_counter = 0
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'best_miou': best_miou,
            'val_metrics': val_metrics,
            'config': {
                'num_classes': 10, 'base_features': CONFIG['base_features'],
                'img_size': list(CONFIG['img_size']), 'batch_size': CONFIG['batch_size'],
                'severity_methodology': CONFIG['severity_method'],
            },
        }, best_ckpt)
        print(f'  --> Best model saved (mIoU={best_miou:.4f})')
    else:
        patience_counter += 1
        if patience_counter >= PATIENCE:
            print(f'Early stopping after {epoch} epochs.')
            break

    # Save last
    torch.save({'epoch': epoch, 'model_state_dict': model.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(), 'best_miou': best_miou},
               CONFIG['save_dir'] / 'floodnet_unet_last.pth')

total_time = time.time() - t_start
print(f'\nTraining done in {total_time/60:.1f} min  |  Best mIoU: {best_miou:.4f}')

In [ ]:
# ── Step 11: Save training history ────────────────────────────────────────
history_path = CONFIG['reports_dir'] / 'training_history.json'
with open(history_path, 'w') as f:
    json.dump({
        'history': history, 'best_miou': best_miou,
        'total_training_seconds': round(total_time, 1),
        'severity_methodology': CONFIG['severity_method'] + '. FloodNet does NOT provide ground-truth severity labels.',
    }, f, indent=2)
print(f'History saved to {history_path}')

In [ ]:
# ── Step 12: Plot training curves ─────────────────────────────────────────
import matplotlib.pyplot as plt

epochs_x = [r['epoch'] for r in history]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs_x, [r['train_loss'] for r in history], label='Train')
axes[0].plot(epochs_x, [r['val_loss']   for r in history], label='Val')
axes[0].set_title('Loss'); axes[0].legend(); axes[0].grid(True)

axes[1].plot(epochs_x, [r['mIoU'] for r in history], color='green')
axes[1].set_title('mIoU'); axes[1].grid(True)

axes[2].plot(epochs_x, [r['road_flooded_iou']     for r in history], label='Road-flooded')
axes[2].plot(epochs_x, [r['road_non_flooded_iou'] for r in history], label='Road-non-flooded')
axes[2].plot(epochs_x, [r['water_iou']            for r in history], label='Water')
axes[2].set_title('Key Class IoU'); axes[2].legend(); axes[2].grid(True)

plt.tight_layout()
plt.savefig(CONFIG['reports_dir'] / 'training_curves.png', dpi=150)
plt.show()
print('Curves saved.')

In [ ]:
# ── Step 13: Download checkpoint & history ─────────────────────────────────
# Option A — Download to local PC:
# from google.colab import files
# files.download(str(best_ckpt))
# files.download(str(history_path))

# Option B — Save to Drive:
import shutil
DRIVE_SAVE = Path('/content/drive/MyDrive/FloodNet_Training_Output')
DRIVE_SAVE.mkdir(exist_ok=True)
shutil.copy(best_ckpt, DRIVE_SAVE / 'floodnet_unet_best.pth')
shutil.copy(history_path, DRIVE_SAVE / 'training_history.json')
print(f'Saved to Google Drive: {DRIVE_SAVE}')